In [1]:
!pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Calling Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LORA_MODEL = "banty1614/codeforge-qwen-lora"   # or "./codeforge_qwen_lora"

# Base Model
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Fine-tuned Model
ft_tokenizer = AutoTokenizer.from_pretrained(LORA_MODEL)

ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

ft_model = PeftModel.from_pretrained(
    ft_base,
    LORA_MODEL,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/120M [00:00<?, ?B/s]

Testing both model

In [3]:
def ask_base(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = base_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = base_tokenizer(
        text,
        return_tensors="pt"
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.2,
            do_sample=False,
        )

    return base_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

In [4]:
def ask_finetuned(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = ft_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = ft_tokenizer(
        text,
        return_tensors="pt"
    ).to(ft_model.device)

    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.2,
            do_sample=False,
        )

    return ft_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

In [5]:
test_prompts = [
    "Write a Python function to reverse a linked list.",
    "Implement merge sort.",
    "Explain decorators with an example.",
    "Write a FastAPI CRUD API.",
    "Explain Python generators.",
    "Implement binary search.",
    "Write a stack using classes.",
    "Detect cycle in linked list.",
    "Explain async programming.",
    "Implement BFS traversal."
]

results = []

for prompt in test_prompts:
    results.append({
        "Prompt": prompt,
        "Base": ask_base(prompt),
        "FineTuned": ask_finetuned(prompt),
    })

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [7]:
for r in results:
    print("=" * 120)
    print("PROMPT")
    print(r["Prompt"])

    print("\nBASE MODEL\n")
    print(r["Base"])

    print("\nFINE-TUNED MODEL\n")
    print(r["FineTuned"])
    print("\n")

PROMPT
Write a Python function to reverse a linked list.

BASE MODEL

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a Python function to reverse a linked list.
assistant
Certainly! Reversing a linked list is a common task in data structures and algorithms. Below is a Python function that reverses a singly linked list:

```python
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def reverse_linked_list(head):
    """
    Reverses a singly linked list.

    :param head: The head node of the linked list.
    :return: The new head of the reversed linked list.
    """
    prev_node = None
    current_node = head
    while current_node:
        # Store the next node
        next_node = current_node.next
        # Reverse the current node's pointer
        current_node.next = prev_node
        # Move pointers forward
        prev_node = current_node
        current_node = next_node
    return pre